# Effective binding × frequency analysis

This notebook reads the effective-binding CSVs. It does **not** load a language
model. It reproduces the late-layer frequency test with raw attention and with
norm-aware binding, saves tables, and selects candidate heads using a fixed
25-compound selection set before evaluation on 24 held-out compounds.


In [ ]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Cell 0a: Colab-only analysis dependencies
if IN_COLAB:
    %pip install -q pandas scipy matplotlib


In [ ]:
# Cell 1: Imports
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

CONDITION = "uniform"   # run once as natural, then once as uniform
N_BOOTSTRAPS = 2000
RANDOM_SEED = 20260813

assert CONDITION in {"natural", "uniform"}

output_dir = PROJECT_ROOT / "results" / "analysis"
figure_dir = PROJECT_ROOT / "results" / "analysis" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)


## Load and validate the CSVs

The error messages in this cell are intentional: if a model run is incomplete,
the notebook stops rather than silently analyzing a partial file.


In [ ]:
from src.effective_binding_inputs import condition_files, load_condition_frame
files = condition_files(PROJECT_ROOT, CONDITION)

if not files:
    raise FileNotFoundError(
        f"No {CONDITION!r} effective-binding files found. "
        "Run an effective-binding model notebook first."
    )

required = {
    "compound", "layer", "head", "attention_weight",
    "weighted_ov_norm", "relative_weighted_ov_norm", "model",
}

frames = []
for path in files:
    frame = load_condition_frame(path, PROJECT_ROOT, CONDITION)
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")
    frame["family"] = path.parts[-4]
    frames.append(frame)
    print(f"✓ {path.parts[-4]:7s} {frame['model'].iloc[0]:22s} {len(frame):,} rows")

detailed = pd.concat(frames, ignore_index=True)
print(f"\nLoaded {len(detailed):,} rows from {len(files)} models.")


## Summarize the final third

`max_attention` directly reproduces the old measurement. The other summaries
ask whether the same pattern survives after incorporating the value/OV write.
The 95th percentile and top-five mean are less dependent on one extreme head.


In [ ]:
def top_five_mean(values):
    return values.nlargest(min(5, len(values))).mean()


# Infer each model's depth from its saved layer numbers.
detailed["n_layers"] = detailed.groupby(
    ["family", "model"]
)["layer"].transform("max") + 1
detailed["first_late_layer"] = np.ceil(
    detailed["n_layers"] * 2 / 3
).astype(int)

late = detailed[detailed["layer"] >= detailed["first_late_layer"]].copy()

summary = (
    late.groupby(["family", "model", "compound"], as_index=False)
        .agg(
            max_attention=("attention_weight", "max"),
            max_weighted=("weighted_ov_norm", "max"),
            max_relative=("relative_weighted_ov_norm", "max"),
            p95_relative=("relative_weighted_ov_norm", lambda x: x.quantile(.95)),
            mean_top5_relative=("relative_weighted_ov_norm", top_five_mean),
        )
)

frequency = pd.read_csv(PROJECT_ROOT / "results" / "frequency" / "frequency_table.csv")
frequency = frequency[["compound", "bigram_count"]].drop_duplicates("compound")
summary = summary.merge(frequency, on="compound", how="inner", validate="many_to_one")
summary["log_frequency"] = np.log1p(summary["bigram_count"])

counts = summary.groupby(["family", "model"])["compound"].nunique()
if (counts < 40).any():
    raise ValueError(f"Too few matched compounds in one or more models:\n{counts}")

summary_path = output_dir / f"effective_binding_compound_summary_{CONDITION}.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")
display(summary.head())


## Model-level correlations and bootstrap intervals

The bootstrap resamples compounds, not individual heads. The important comparison
is whether the negative frequency relationship survives for norm-aware measures.


In [ ]:
MEASURES = [
    "max_attention", "max_weighted", "max_relative",
    "p95_relative", "mean_top5_relative",
]


def bootstrap_spearman(x, y, n_boot=N_BOOTSTRAPS, seed=RANDOM_SEED):
    x = np.asarray(x)
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        chosen = rng.integers(0, len(x), len(x))
        rho = spearmanr(x[chosen], y[chosen]).statistic
        if np.isfinite(rho):
            estimates.append(rho)
    if not estimates:
        return np.array([np.nan, np.nan])
    return np.quantile(estimates, [.025, .975])


rows = []
for (family, model_name), group in summary.groupby(["family", "model"]):
    for measure in MEASURES:
        test = spearmanr(group["log_frequency"], group[measure])
        low, high = bootstrap_spearman(group["log_frequency"], group[measure])
        rows.append({
            "condition": CONDITION,
            "family": family,
            "model": model_name,
            "measure": measure,
            "n_compounds": len(group),
            "rho": test.statistic,
            "p_value": test.pvalue,
            "ci_low": low,
            "ci_high": high,
        })

correlations = pd.DataFrame(rows)
corr_path = output_dir / f"effective_binding_correlations_{CONDITION}.csv"
correlations.to_csv(corr_path, index=False)

display(
    correlations.pivot_table(
        index=["family", "model"], columns="measure", values="rho"
    ).round(3)
)
print(f"Saved: {corr_path}")


## Preregistered aggregate permutation test

This test uses all 49 compounds. For each permutation, the same shuffled frequency labels are used for every model. The aggregate statistic is the mean Fisher-transformed Spearman correlation across models, and the one-sided alternative is that effective binding decreases as frequency increases.


In [ ]:
N_PERMUTATIONS = 10_000
PRIMARY_MEASURE = "p95_relative"

compound_order = sorted(summary["compound"].unique())
if len(compound_order) != 49:
    raise ValueError(
        f"Expected 49 compounds for the preregistered test, found {len(compound_order)}."
    )

model_order = list(
    summary[["family", "model"]]
    .drop_duplicates()
    .sort_values(["family", "model"])
    .itertuples(index=False, name=None)
)
if len(model_order) != 13:
    raise ValueError(
        f"Expected 13 models for the preregistered test, found {len(model_order)}."
    )
frequency_by_compound = (
    summary[["compound", "log_frequency"]]
    .drop_duplicates("compound")
    .set_index("compound")
    .loc[compound_order, "log_frequency"]
    .to_numpy()
)

measure_by_model = []
observed_rhos = []
for family, model_name in model_order:
    group = (
        summary[(summary["family"] == family) & (summary["model"] == model_name)]
        .set_index("compound")
        .loc[compound_order]
    )
    values = group[PRIMARY_MEASURE].to_numpy()
    if len(values) != len(compound_order) or not np.isfinite(values).all():
        raise ValueError(f"Incomplete values for {family}/{model_name}.")
    measure_by_model.append(values)
    observed_rhos.append(spearmanr(frequency_by_compound, values).statistic)

observed_rhos = np.asarray(observed_rhos, dtype=float)
if not np.isfinite(observed_rhos).all():
    raise ValueError("At least one model has an undefined observed correlation.")
observed_statistic = float(np.mean(np.arctanh(observed_rhos)))

permutation_rng = np.random.default_rng(RANDOM_SEED)
extreme_count = 0
for _ in range(N_PERMUTATIONS):
    shuffled_frequency = permutation_rng.permutation(frequency_by_compound)
    permuted_rhos = [
        spearmanr(shuffled_frequency, values).statistic
        for values in measure_by_model
    ]
    permuted_statistic = float(np.mean(np.arctanh(permuted_rhos)))
    if permuted_statistic <= observed_statistic:
        extreme_count += 1

permutation_p = (extreme_count + 1) / (N_PERMUTATIONS + 1)
permutation_result = pd.DataFrame([{
    "condition": CONDITION,
    "measure": PRIMARY_MEASURE,
    "n_models": len(model_order),
    "n_compounds": len(compound_order),
    "n_permutations": N_PERMUTATIONS,
    "random_seed": RANDOM_SEED,
    "observed_mean_fisher_z": observed_statistic,
    "extreme_permutation_count": extreme_count,
    "one_sided_p_value": permutation_p,
}])
permutation_path = (
    output_dir / f"effective_binding_aggregate_permutation_{CONDITION}.csv"
)
permutation_result.to_csv(permutation_path, index=False)
display(permutation_result)
print(f"Saved: {permutation_path}")


In [ ]:
# Figure: raw attention versus norm-aware binding
plot_data = correlations[
    correlations["measure"].isin(["max_attention", "p95_relative"])
].copy()

fig, ax = plt.subplots(figsize=(11, 5))
labels = (plot_data[["family", "model"]].drop_duplicates()
          .assign(label=lambda x: x["family"] + ":" + x["model"]))
order = labels["label"].tolist()
x = np.arange(len(order))

for offset, (measure, label, color) in zip(
    [-0.18, 0.18],
    [
        ("max_attention", "Raw attention maximum", "#777777"),
        ("p95_relative", "Norm-aware binding (95th percentile)", "#276FBF"),
    ],
):
    sub = plot_data[plot_data["measure"] == measure].copy()
    sub["label"] = sub["family"] + ":" + sub["model"]
    sub = sub.set_index("label").reindex(order)
    ax.bar(x + offset, sub["rho"], width=.34, label=label, color=color)

ax.axhline(0, color="black", linewidth=.8)
ax.set_ylabel("Spearman correlation with log bigram frequency")
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha="right")
ax.legend(frameon=False)
ax.set_title(f"Late-layer frequency relationship — {CONDITION} prompts")
fig.tight_layout()
figure_path = figure_dir / f"effective_binding_frequency_{CONDITION}.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {figure_path}")


## Select heads without looking at the test compounds

The split is made once from compound names and reused for every model. Candidate
heads are chosen using the 25 selection compounds only. Their frequency
relationship is then measured on the 24 untouched held-out compounds.


In [ ]:
all_compounds = sorted(summary["compound"].unique())
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(all_compounds)
if len(all_compounds) != 49:
    raise ValueError(
        f"Expected 49 compounds for the preregistered split, found {len(all_compounds)}."
    )
split_point = 25
selection_compounds = set(all_compounds[:split_point])
test_compounds = set(all_compounds[split_point:])

assert len(selection_compounds) == 25
assert len(test_compounds) == 24
assert selection_compounds.isdisjoint(test_compounds)
assert selection_compounds | test_compounds == set(all_compounds)

split_table = pd.DataFrame({
    "compound": all_compounds,
    "split": ["selection"] * split_point + ["held_out"] * (len(all_compounds) - split_point),
    "shuffle_position": np.arange(1, len(all_compounds) + 1),
    "random_seed": RANDOM_SEED,
})
split_path = output_dir / "effective_binding_compound_split.csv"
split_table.to_csv(split_path, index=False)

print(f"Selection compounds: {len(selection_compounds)}")
print(f"Test compounds:      {len(test_compounds)}")
print(f"Saved: {split_path}")

late_with_frequency = late.merge(
    frequency, on="compound", how="inner", validate="many_to_one"
)
late_with_frequency["log_frequency"] = np.log1p(
    late_with_frequency["bigram_count"]
)


def benjamini_hochberg(p_values):
    """False-discovery-rate adjusted p-values, implemented without extra packages."""
    p = np.asarray(p_values, dtype=float)
    result = np.full(p.shape, np.nan, dtype=float)
    finite_positions = np.flatnonzero(np.isfinite(p))
    if len(finite_positions) == 0:
        return result

    # Constant-signal heads have undefined Spearman tests (NaN). Exclude only
    # those heads from the correction; otherwise one NaN contaminates every
    # adjusted p-value in the model.
    finite_p = p[finite_positions]
    order = np.argsort(finite_p)
    ranked = finite_p[order]
    adjusted = ranked * len(finite_p) / np.arange(1, len(finite_p) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    finite_result = np.empty_like(adjusted)
    finite_result[order] = np.clip(adjusted, 0, 1)
    result[finite_positions] = finite_result
    return result


head_rows = []
test_rows = []
candidate_rows = []

for (family, model_name), model_data in late_with_frequency.groupby(["family", "model"]):
    selection = model_data[model_data["compound"].isin(selection_compounds)]
    per_head = []
    for (layer, head), group in selection.groupby(["layer", "head"]):
        test = spearmanr(group["log_frequency"], group["relative_weighted_ov_norm"])
        per_head.append({
            "condition": CONDITION, "family": family, "model": model_name,
            "layer": layer, "head": head, "selection_rho": test.statistic,
            "selection_p": test.pvalue, "n_selection": len(group),
        })

    per_head = pd.DataFrame(per_head)
    per_head["selection_p_fdr"] = benjamini_hochberg(per_head["selection_p"])
    per_head = per_head.sort_values("selection_rho")
    head_rows.append(per_head)

    candidates = per_head.head(5).copy()
    candidates["candidate_rank"] = np.arange(1, len(candidates) + 1)
    candidate_rows.append(candidates)

    pairs = list(zip(candidates["layer"].astype(int), candidates["head"].astype(int)))
    held_out = model_data[
        model_data["compound"].isin(test_compounds)
        & model_data.set_index(["layer", "head"]).index.isin(pairs)
    ]
    held_out_scores = held_out.groupby("compound", as_index=False).agg(
        candidate_head_signal=("relative_weighted_ov_norm", "mean"),
        log_frequency=("log_frequency", "first"),
    )
    test = spearmanr(
        held_out_scores["log_frequency"],
        held_out_scores["candidate_head_signal"],
    )
    test_rows.append({
        "condition": CONDITION, "family": family, "model": model_name,
        "n_test": len(held_out_scores), "held_out_rho": test.statistic,
        "held_out_p": test.pvalue,
    })

head_results = pd.concat(head_rows, ignore_index=True)
candidates = pd.concat(candidate_rows, ignore_index=True)
held_out_results = pd.DataFrame(test_rows)

head_results.to_csv(
    output_dir / f"effective_binding_all_head_correlations_{CONDITION}.csv",
    index=False,
)
candidates.to_csv(
    output_dir / f"effective_binding_head_candidates_{CONDITION}.csv",
    index=False,
)
held_out_results.to_csv(
    output_dir / f"effective_binding_head_heldout_{CONDITION}.csv",
    index=False,
)

print("Candidate heads (chosen on selection compounds only):")
display(candidates[[
    "family", "model", "candidate_rank", "layer", "head",
    "selection_rho", "selection_p_fdr",
]])
print("Held-out compound results:")
display(held_out_results)


In [ ]:
# Optional: commit only the tables and figures created by this notebook.
import os
RUN_GIT_SAVE = False

if RUN_GIT_SAVE:
    os.chdir(PROJECT_ROOT)
    !git config user.email "trisha@trishasalas.com"
    !git config user.name "Trisha Salas"
    !git add results/analysis/
    !git commit -m "analyze effective binding and frequency: {CONDITION}"
    !git push
else:
    print("Git save skipped. Review generated files before committing.")
